# langgraph结构图


In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END

# ==========================================
# 1. 定义状态 (Mock AgentState)
# ==========================================
class AgentState(TypedDict):
    # 这里定义你工作流中传递的数据字段
    # 例如：requirements, plan, budget, preference, feedback等
    user_input: str
    plan: str
    budget: str
    preference: str
    human_feedback: str

# ==========================================
# 2. 定义各个节点函数 (Mock Agents)
# ==========================================
def requirement_agent(state: AgentState):
    print("-> 执行 requirement_agent")
    return {"user_input": "用户需求已收集"}

def planner_agent(state: AgentState):
    print("-> 执行 planner_agent")
    return {"plan": "新的计划已生成"}

def budget_agent(state: AgentState):
    print("-> 执行 budget_agent")
    return {"budget": "预算已计算"}

def preference_agent(state: AgentState):
    print("-> 执行 preference_agent")
    return {"preference": "偏好已分析"}

def human_review_agent(state: AgentState):
    print("-> 执行 human_review_agent (等待人工审核)")
    # 实际应用中，这里可能是中断等待人工输入
    return {"human_feedback": "人工审核通过"}

# ==========================================
# 3. 定义路由函数 (Mock 路由逻辑)
# ==========================================
def supervisor(state: AgentState) -> Literal["human_review", "planner"]:
    """
    根据 preference 节点的结果决定下一步。
    返回的字符串必须与 path_map 中的键匹配。
    """
    # 这里写你的实际业务逻辑
    # 为了演示，假设每次都进入人工审核
    return "human_review"

def should_continue_after_human(state: AgentState) -> Literal["end", "planner"]:
    """
    根据 human_review 的结果决定是结束还是打回 planner。
    """
    # 这里写你的实际业务逻辑
    feedback = state.get("human_feedback", "")
    if "通过" in feedback:
        return "end"
    else:
        return "planner"

# ==========================================
# 4. 构建工作流 (你提供的核心代码)
# ==========================================
def build_workflow():
    """构建并编译工作流"""
    workflow = StateGraph(AgentState)

    # ========== 1. 添加节点 ==========
    workflow.add_node("requirement", requirement_agent)
    workflow.add_node("planner", planner_agent)
    workflow.add_node("budget", budget_agent)
    workflow.add_node("preference", preference_agent)
    workflow.add_node("human_review", human_review_agent)
    # 注意: supervisor 不作为节点添加，它只作为路由函数使用

    # ========== 2. 入口 ==========
    workflow.set_entry_point("requirement")

    # ========== 3. 线性链 ==========
    workflow.add_edge(start_key="requirement", end_key="planner")
    workflow.add_edge(start_key="planner", end_key="budget")
    workflow.add_edge(start_key="budget", end_key="preference")

    # ========== 4. supervisor 条件边（从 preference 出发） ==========
    workflow.add_conditional_edges(
        source="preference",
        path=supervisor,  # 路由函数，返回字符串
        path_map={
            "human_review": "human_review",
            "planner": "planner",  # 审核未过 -> 回 planner
        }
    )

    # ========== 5. human_review 条件边 ==========
    workflow.add_conditional_edges(
        source="human_review",
        path=should_continue_after_human,  # 路由函数
        path_map={
            "end": END,          # 人工通过 -> 结束
            "planner": "planner" # 人工拒绝 -> 回 planner
        }
    )

    # 编译工作流
    app = workflow.compile()
    return app

# 构建并获取编译后的 app
app = build_workflow()
print("工作流构建成功！")

工作流构建成功！


In [4]:
from IPython.display import display, Markdown

# 获取 Mermaid 格式的代码
mermaid_code = app.get_graph().draw_mermaid()
# 使用 Markdown 渲染 Mermaid
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	requirement(requirement)
	planner(planner)
	budget(budget)
	preference(preference)
	human_review(human_review)
	__end__([<p>__end__</p>]):::last
	__start__ --> requirement;
	budget --> preference;
	human_review -. &nbsp;end&nbsp; .-> __end__;
	human_review -.-> planner;
	planner --> budget;
	preference -.-> human_review;
	preference -.-> planner;
	requirement --> planner;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```